# Analiza dostępności i usług sklepów Żabka w Polsce

Autorzy:
- Jakub Rosiak 251620,
- Mateusz Kosowski 251558,
- Nikodem Nowak 251598

Nasz projekt koncentruje się na analizie rozmieszczenia sieci sklepów Żabka w Polsce, wykorzystując zbiór danych o lokalizacji prawie 10 tysięcy placówek (stan na rok 2024) oraz dane demograficzne z Narodowego Spisu Powszechnego 2021 (siatka kilometrowa GUS). Poprzez integrację danych punktowych z warstwą demograficzną, projekt pozwala na wyznaczenie obszarów o wysokim potencjale inwestycyjnym. Rezultatem prac jest zestaw interaktywnych wizualizacji oraz wniosków biznesowych wspierających procesy decyzyjne w zakresie ekspansji sieci.

### Cele projektu:
- <b>Prezentacja danych statystycznych sieci:</b>
    - Analiza struktury usług dodatkowych
    - Ranking województw i miast pod względem liczby placówek.
    - Badanie korelacji między liczbą mieszkańców a liczbą placówek.
- <b>Analiza przestrzenna i demograficzna:</b>
    - Wizualizacja rozmieszczenia sklepów na mapie Polski
    - Wizualizacja liczby mieszkańców przypadających na jeden sklep w siatce kilometrowej.
    - Oszacowanie liczby Polaków posiadających sklep w bezpośrednim sąsiedztwie (analiza dostępności w oparciu o siatkę).
- <b>Wnioski biznesowe:</b>
    - Wskazanie obszarów o najwyższym potencjale inwestystycyjnym




In [23]:
# Importy
import os
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster, HeatMap
from scipy.spatial import cKDTree
from scipy import stats
from branca.element import MacroElement
from jinja2 import Template


# Konfiguracja wyglądu Seaborn i Matplotlib
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333',
    'grid.color': '#ddd',
    'text.color': '#222',
    'figure.figsize': (14, 8)
})

# Tworzenie katalogu na output
os.makedirs("output", exist_ok=True)

# Ignorujemy ostrzeżenia aby konsola była czysta
warnings.filterwarnings('ignore')

W naszym projekcie posługujemy się dwoma różnymi systemami współrzędnych przestrzennych (CRS) definiowanych przez standard EPSG.

<b>EPSG:4326</b> to globalny system układu współrzędnych geograficznych, w którym jednostką są stopnie (długość i szerokość geograficzna). W naszym projekcie służy on do:

   1. Wizualizacji interaktywnej: Jest to standardowy układ obsługiwany przez bibliotekę Folium
   2. Przechowywania surowych danych: Współrzędne sklepów w pliku wejściowym zapisane są jako szerokość (lat) i długość (lng).

Natomiast <b> EPSG:2180</b> to system prostokątnych współrzędnych płaskich zaprojektowany specjalnie dla obszaru Polski. W naszym projekcie służy on do:
   1. Precyzyjnych obliczeń odległości: Ponieważ jednostką w tym układzie jest metr, pozwala on na dokładne wyliczenie dystansu między mieszkańcami a najbliższym sklepem Żabka
   2. Złączeń przestrzennych (Spatial Join): Umożliwia poprawne przypisanie sklepów do komórek siatki populacyjnej GUS, która natywnie korzysta z tego formatu.
   3. Analizy zagęszczenia: Jest niezbędny do rzetelnego operowania na danych o populacji

In [24]:
# Wczytywanie danych
df_shops = pd.read_csv("data/zabka_shops.csv")
df_shops = df_shops[
    (df_shops['lat'].between(49, 55)) &
    (df_shops['lng'].between(14, 25))
].copy()
df_shops['services'] = df_shops['services'].fillna('').astype(str)

# Przygotowanie GeoDataFrame w układzie 4326 dla Folium
gdf_shops_4326 = gpd.GeoDataFrame(
    df_shops,
    geometry=gpd.points_from_xy(df_shops.lng, df_shops.lat),
    crs="EPSG:4326"
)

# # Siatka GUS z populacją
try:
    gdf_population = gpd.read_file("data/GRID_NSP2021_RES/GRID_NSP2021_RES.shp")
    gdf_population.set_crs(epsg=2180, allow_override=True, inplace=True)
    gdf_shops_2180 = gdf_shops_4326.to_crs(epsg=2180)
    print(f"✓ Siatka GUS: {len(gdf_population)} komórek")
except Exception as e:
    print(f"✗ Błąd wczytywania siatki GUS: {e}")
    raise SystemExit(1)


Liczba sklepów Żabka w Polsce (2024r.): 9755
Siatka wczytana. Układ: EPSG:2180


Musimy dokonać pewnych obliczeń przestrzennych, potrzebnych do wykresów i warstw mapy

In [ ]:
# Sklepy w komórkach siatki
joined = gpd.sjoin(gdf_shops_2180, gdf_population, how="inner", predicate="within")
shop_counts = joined['index_right'].value_counts()
gdf_population['shop_count'] = 0
gdf_population.loc[shop_counts.index, 'shop_count'] = shop_counts

# Dystans do najbliższej Żabki (KDTree - szybki)
shop_coords = np.column_stack([
    gdf_shops_2180.geometry.x,
    gdf_shops_2180.geometry.y
])
tree = cKDTree(shop_coords)
centroids = gdf_population.geometry.centroid
grid_coords = np.column_stack([centroids.x, centroids.y])
distances, _ = tree.query(grid_coords)
gdf_population['distance_m'] = distances

# Kategorie dystansu
bins = [0, 1000, 2000, 5000, np.inf]
labels = ['< 1km', '1-2km', '2-5km', '> 5km']
gdf_population['distance_cat'] = pd.cut(gdf_population['distance_m'], bins=bins, labels=labels)

# Ludzie na sklep (tylko tam gdzie są sklepy) - do choropleth
mask_shops = gdf_population['shop_count'] > 0
gdf_population['people_per_shop'] = np.nan
gdf_population.loc[mask_shops, 'people_per_shop'] = (
    gdf_population.loc[mask_shops, 'RES'] / gdf_population.loc[mask_shops, 'shop_count']
)

# Obszary niedostępne - duży potencjał inwestycyjny (>= 500 osób i > 1.5 km)
underserved = gdf_population[
    (gdf_population['RES'] >= 500) &
    (gdf_population['distance_m'] > 1500)
].copy()
underserved['priority'] = underserved['RES'] * underserved['distance_m'] / 1000
underserved = underserved.sort_values('priority', ascending=False)

# Przypisanie województw
_, nearest_shop_idx = tree.query(grid_coords)
gdf_population['voivodeship'] = df_shops.iloc[nearest_shop_idx]['voivodeship'].values
